In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import importlib
import moderate
import random

from test_data.spam import spam_profile_descriptions
from test_data.gibberish import gibberish_profile_descriptions
from test_data.inappropriate import inappropriate_profile_descriptions
from test_data.unsupported_languages import unsupported_language_profile_descriptions
from test_data.valid import valid_profile_descriptions

importlib.reload(moderate)

agent = moderate.initialize_agent()

# select categories to evaluate
categories = [
    "valid",
    "spam",
    "gibberish",
    "inappropriate",
    "language",
]

test_data = []

for category in categories:
    if category == "valid":
        for description in valid_profile_descriptions:
            test_data.append((description, "is_valid"))
    elif category == "spam":
        for description in spam_profile_descriptions:
            test_data.append((description, "is_spam"))
    elif category == "gibberish":
        for description in gibberish_profile_descriptions:
            test_data.append((description, "is_gibberish"))
    elif category == "inappropriate":
        for description in inappropriate_profile_descriptions:
            test_data.append((description, "is_inappropriate"))
    elif category == "language":
        for description in unsupported_language_profile_descriptions:
            test_data.append((description, "language"))

# retrieve items randomly from test_data
random.shuffle(test_data)
test_data = test_data[:4]

count_correct = 0
count_total = 0

incorrectly_classified = []

for profile, flag in test_data:
    result = moderate.moderate_profile_description(profile, agent)
    print(f"Ad: {profile}")
    print(f"Result: {result}")

    count_total += 1

    if flag != "language":
        if result[flag] is True:
            count_correct += 1
            print(
                f"Correctly evaluated as {flag}: {count_correct} out of {count_total}"
            )
        else:
            incorrectly_classified.append((profile, result))
    else:
        if result["language"] not in moderate.accepted_languages:
            count_correct += 1
            print(
                f"Correctly evaluated with language {result['language']}: {count_correct} out of {count_total}"
            )
        else:
            incorrectly_classified.append((profile, result))

    print("---")

print(f"Total correctly evaluated: {count_correct} out of {count_total}")
print("---")

for profile, result in incorrectly_classified:
    print(f"Incorrectly classified: {profile}")
    print(f"Result: {result}")
    print("---")

Using model: google_genai:gemini-3-flash-preview
Ad: Idraulico disponibile per riparazioni rapide in cucina e bagno. Eseguo sostituzione rubinetti, sifoni, scarichi e piccoli interventi con attenzione alla pulizia finale.
Result: {'is_valid': False, 'is_gibberish': False, 'is_spam': False, 'is_inappropriate': False, 'language': 'italian', 'confidence': 1.0, 'reason': 'Professional plumbing service offer in Italian.'}
Correctly evaluated with language italian: 1 out of 1
---
Ad: Pavator pentru curti rezidentiale, alei, parcari si zone de acces. Fac trasare, pregatirea straturilor suport, montaj borduri si compactare, astfel incat suprafata sa nu joace in timp.
Result: {'is_valid': True, 'is_gibberish': False, 'is_spam': False, 'is_inappropriate': False, 'language': 'romanian', 'confidence': 1.0, 'reason': 'Professional and relevant service description in Romanian.'}
Correctly evaluated as is_valid: 2 out of 2
---
Ad: Nu mai cauta mesteri aici, intra pe linkul meu si primesti reduceri ga